# Screws

**Type:** Producer — stores output in dimensional depot  
**Blueprint:** `screw` (40 Iron Ingots/min → 160 Screws/min at 100%)  
**Internal chain:** Ingots → Iron Rods (3 constructors) → Screws (4 constructors)  
**Downstream:** Rotors and Reinforced Iron Plates both pull from this storage

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS
import pandas as pd

BP  = BLUEPRINTS
TOL = 0.5


def machine_hover(bp: Blueprint, output_key: str, target_rate: float) -> str:
    if not bp.stages:
        return ""
    rates = bp.stage_rates(output_key, target_rate)
    lines = ["── Machines (per copy) ──"]
    for r in rates:
        name = ITEMS[r["item_key"]].name
        lines.append(f"  {r['machines']}× {r['building'].title()} → {r['per_machine_rate']:.2f} {name}/min each")
    return "<br>" + "<br>".join(lines)

## Constants

Set `IRON_INGOTS` to your allocation from the 270/min supply.  
Set `STASH_RATE` to how many Screws/min should accumulate in storage — Rotors and Reinforced Iron Plates pull the remainder.

> **Note:** Two downstream notebooks (Rotors and Reinforced Iron Plates) both consume from this storage.
> Their combined pull must not exceed `screw_total − STASH_RATE`.

In [ ]:
# ── Supply ────────────────────────────────────────────────────────────────────
IRON_INGOTS = None  # ← your ingot allocation (items/min)

# ── Stash target ──────────────────────────────────────────────────────────────
STASH_RATE = 50.0   # Screws/min accumulating in storage

# ── Blueprint placement ───────────────────────────────────────────────────────
SCREW_COPIES      = None  # ← number of blueprints to place
SCREW_OUTPUT_EACH = None  # ← output rate per copy to set in-game (Screws/min)

## Derived rates and balance

In [ ]:
assert None not in (IRON_INGOTS, SCREW_COPIES, SCREW_OUTPUT_EACH), \
    "Fill in all constants in the cell above before running this cell."

screw_total  = SCREW_COPIES * SCREW_OUTPUT_EACH
screw_ingots = screw_total  * BP['screw'].input_ratio('iron-ingot', 'screw')

available_for_downstream = screw_total - STASH_RATE

assert abs(screw_ingots - IRON_INGOTS) < TOL, (
    f"Ingot balance: consuming {screw_ingots:.2f}/min, supplied {IRON_INGOTS:.0f}/min "
    f"(delta {screw_ingots - IRON_INGOTS:+.2f})"
)
assert available_for_downstream > 0, (
    f"Not enough screws to stash {STASH_RATE}/min — only producing {screw_total:.2f}/min"
)

# Internal chain rates (per blueprint copy)
internal = BP['screw'].stage_rates('screw', SCREW_OUTPUT_EACH)

print(f"✓ Chain balanced")
print(f"  {IRON_INGOTS:.0f} Iron Ingots/min  →  {screw_total:.2f} Screws/min")
print(f"  Stashing:                        {STASH_RATE:.0f} Screws/min")
print(f"  Available for downstream:        {available_for_downstream:.2f} Screws/min")
print()
print(f"  Per blueprint copy (set in-game):")
for r in internal:
    name = ITEMS[r['item_key']].name
    print(f"    {r['machines']}× Constructor → {r['per_machine_rate']:.2f} {name}/min each")